In [ ]:
"""
Inception-v4 Transfer Learning for Brain Tumor Classification
Based on: Ishfaq et al. (2025) - Scientific Reports

IMPORTANT: This version uses ALREADY PREPROCESSED AND AUGMENTED data
No additional augmentation is applied during training

Model Specifications:
- Base: Inception-v4 (InceptionResNetV2)
- Input size: 299x299x3
- Fine-tuning: All layers trainable
- Output: 10 classes
- Expected Test Accuracy: 99.3%
"""

import os
import numpy as np
import matplotlib.pyplot as plt
import json
from datetime import datetime
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import InceptionResNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns


class InceptionV4Classifier:
    def __init__(self, input_shape=(299, 299, 3), num_classes=10):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.model = None
        self.history = None
        self.class_names = None

    def build_model(self, trainable=True):
        base_model = InceptionResNetV2(
            include_top=False,
            weights='imagenet',
            input_shape=self.input_shape,
            pooling='avg'
        )

        base_model.trainable = trainable

        inputs = keras.Input(shape=self.input_shape)
        x = base_model(inputs, training=trainable)
        
        x = layers.Dropout(0.5)(x)
        x = layers.Dense(512, activation='relu', name='fc1')(x)
        x = layers.Dropout(0.3)(x)
        outputs = layers.Dense(self.num_classes, activation='softmax', name='predictions')(x)

        self.model = keras.Model(inputs, outputs, name='InceptionV4_Classifier')
        return self.model

    def compile_model(self, learning_rate=0.0001):
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

        self.model.compile(
            optimizer=optimizer,
            loss='categorical_crossentropy',
            metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
        )

        print("✅ Inception-v4 model compiled successfully")
        print(f"   Optimizer: Adam (lr={learning_rate})")

    def get_model_summary(self):
        return self.model.summary()

    def count_parameters(self):
        total_params = self.model.count_params()
        trainable_params = sum([np.prod(v.shape) for v in self.model.trainable_weights])

        print(f"\n📊 Model Parameters:")
        print(f"   Total parameters: {total_params:,}")
        print(f"   Trainable parameters: {trainable_params:,}")
        return total_params, trainable_params


class BrainTumorTrainer:
    def __init__(self, data_dir, model, batch_size=64, image_size=(299, 299)):
        self.data_dir = Path(data_dir)
        self.model = model
        self.batch_size = batch_size
        self.image_size = image_size

        self.train_dir = self.data_dir / 'train'
        self.val_dir = self.data_dir / 'val'
        self.test_dir = self.data_dir / 'test'

        self.train_generator = None
        self.val_generator = None
        self.test_generator = None

    def setup_data_generators(self):
        """Setup data generators WITHOUT augmentation (data already augmented)"""
        print("\n" + "=" * 80)
        print("📊 SETTING UP DATA GENERATORS")
        print("=" * 80)
        print("⚠️  Using PRE-AUGMENTED data - No additional augmentation applied")
        print(f"⚠️  Image size: {self.image_size[0]}x{self.image_size[1]} (Inception-v4 requires 299x299)")
        print("=" * 80)

        train_datagen = ImageDataGenerator(rescale=1./255)
        val_datagen = ImageDataGenerator(rescale=1./255)
        test_datagen = ImageDataGenerator(rescale=1./255)

        self.train_generator = train_datagen.flow_from_directory(
            self.train_dir,
            target_size=self.image_size,
            batch_size=self.batch_size,
            class_mode='categorical',
            shuffle=True,
            seed=42
        )

        self.val_generator = val_datagen.flow_from_directory(
            self.val_dir,
            target_size=self.image_size,
            batch_size=self.batch_size,
            class_mode='categorical',
            shuffle=False
        )

        self.test_generator = test_datagen.flow_from_directory(
            self.test_dir,
            target_size=self.image_size,
            batch_size=self.batch_size,
            class_mode='categorical',
            shuffle=False
        )

        self.model.class_names = list(self.train_generator.class_indices.keys())

        print(f"\n📈 Dataset Statistics:")
        print(f"   Training samples: {self.train_generator.samples}")
        print(f"   Validation samples: {self.val_generator.samples}")
        print(f"   Test samples: {self.test_generator.samples}")
        print(f"   Number of classes: {self.train_generator.num_classes}")
        print(f"   Class names: {self.model.class_names}")
        print("=" * 80)

    def train(self, epochs=20, save_dir='models'):
        save_dir = Path(save_dir)
        save_dir.mkdir(exist_ok=True, parents=True)

        print("\n" + "=" * 80)
        print("🚀 STARTING TRAINING - Inception-v4")
        print("=" * 80)
        print(f"Epochs: {epochs}")
        print(f"Batch size: {self.batch_size}")
        print(f"Save directory: {save_dir}")
        print("=" * 80)

        callbacks = [
            ModelCheckpoint(
                filepath=str(save_dir / 'inception_v4_best.h5'),
                monitor='val_accuracy',
                save_best_only=True,
                mode='max',
                verbose=1
            ),
            EarlyStopping(
                monitor='val_loss',
                patience=8,
                restore_best_weights=True,
                verbose=1
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=3,
                min_lr=1e-7,
                verbose=1
            ),
            CSVLogger(
                filename=str(save_dir / 'training_history.csv'),
                separator=',',
                append=False
            )
        ]

        self.model.history = self.model.model.fit(
            self.train_generator,
            epochs=epochs,
            validation_data=self.val_generator,
            callbacks=callbacks,
            verbose=1
        )

        print("\n✅ Training completed!")

        final_model_path = save_dir / 'inception_v4_final.h5'
        self.model.model.save(final_model_path)
        print(f"💾 Final model saved to: {final_model_path}")

        return self.model.history

    def plot_training_history(self, save_dir='models'):
        history = self.model.history.history
        save_dir = Path(save_dir)

        fig, axes = plt.subplots(2, 2, figsize=(15, 10))

        axes[0, 0].plot(history['accuracy'], label='Train', linewidth=2)
        axes[0, 0].plot(history['val_accuracy'], label='Val', linewidth=2)
        axes[0, 0].set_title('Inception-v4: Accuracy', fontsize=14, fontweight='bold')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Accuracy')
        axes[0, 0].legend()
        axes[0, 0].grid(alpha=0.3)

        axes[0, 1].plot(history['loss'], label='Train', linewidth=2)
        axes[0, 1].plot(history['val_loss'], label='Val', linewidth=2)
        axes[0, 1].set_title('Inception-v4: Loss', fontsize=14, fontweight='bold')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Loss')
        axes[0, 1].legend()
        axes[0, 1].grid(alpha=0.3)

        axes[1, 0].plot(history['precision'], label='Train', linewidth=2)
        axes[1, 0].plot(history['val_precision'], label='Val', linewidth=2)
        axes[1, 0].set_title('Inception-v4: Precision', fontsize=14, fontweight='bold')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Precision')
        axes[1, 0].legend()
        axes[1, 0].grid(alpha=0.3)

        axes[1, 1].plot(history['recall'], label='Train', linewidth=2)
        axes[1, 1].plot(history['val_recall'], label='Val', linewidth=2)
        axes[1, 1].set_title('Inception-v4: Recall', fontsize=14, fontweight='bold')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Recall')
        axes[1, 1].legend()
        axes[1, 1].grid(alpha=0.3)

        plt.tight_layout()
        plot_path = save_dir / 'training_history.png'
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        print(f"📊 Training history saved to: {plot_path}")
        plt.show()

    def evaluate_on_test(self, save_dir='models'):
        print("\n" + "=" * 80)
        print("🧪 EVALUATING ON TEST SET - Inception-v4")
        print("=" * 80)

        save_dir = Path(save_dir)

        test_loss, test_accuracy, test_precision, test_recall = self.model.model.evaluate(
            self.test_generator,
            verbose=1
        )

        test_f1 = 2 * (test_precision * test_recall) / (test_precision + test_recall + 1e-7)

        print(f"\n📊 Test Results:")
        print(f"   Accuracy:  {test_accuracy*100:.2f}%")
        print(f"   Precision: {test_precision*100:.2f}%")
        print(f"   Recall:    {test_recall*100:.2f}%")
        print(f"   F1-Score:  {test_f1*100:.2f}%")
        print(f"   Loss:      {test_loss:.4f}")

        print("\n🔮 Generating predictions...")
        self.test_generator.reset()
        predictions = self.model.model.predict(self.test_generator, verbose=1)
        y_pred = np.argmax(predictions, axis=1)
        y_true = self.test_generator.classes

        print("\n📋 Classification Report:")
        print("=" * 80)
        report = classification_report(y_true, y_pred, target_names=self.model.class_names, digits=4)
        print(report)

        report_path = save_dir / 'classification_report.txt'
        with open(report_path, 'w') as f:
            f.write("Inception-v4 Classification Report\n" + "=" * 80 + "\n" + report)
        print(f"💾 Classification report saved to: {report_path}")

        cm = confusion_matrix(y_true, y_pred)

        plt.figure(figsize=(12, 10))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
                    xticklabels=self.model.class_names,
                    yticklabels=self.model.class_names,
                    cbar_kws={'label': 'Count'})
        plt.title('Confusion Matrix - Inception-v4', fontsize=16, fontweight='bold')
        plt.ylabel('True Label', fontsize=12)
        plt.xlabel('Predicted Label', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        plt.tight_layout()

        cm_path = save_dir / 'confusion_matrix.png'
        plt.savefig(cm_path, dpi=300, bbox_inches='tight')
        print(f"📊 Confusion matrix saved to: {cm_path}")
        plt.show()

        results = {
            'model': 'Inception-v4',
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'test_accuracy': float(test_accuracy),
            'test_precision': float(test_precision),
            'test_recall': float(test_recall),
            'test_f1_score': float(test_f1),
            'test_loss': float(test_loss),
            'confusion_matrix': cm.tolist(),
            'class_names': self.model.class_names
        }

        results_path = save_dir / 'test_results.json'
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"💾 Test results saved to: {results_path}")
        print("=" * 80)

        return test_accuracy, test_precision, test_recall, test_f1


def main():
    print("\n" + "=" * 80)
    print("🧠 Inception-v4 - BRAIN TUMOR CLASSIFICATION")
    print("Based on: Ishfaq et al. (2025) - Scientific Reports")
    print("Using PRE-AUGMENTED Data")
    # print("Expected Test Accuracy: 99.3%")
    print("=" * 80)

    DATA_DIR = '/kaggle/input/datasets/fazilsiddiqui/brain-tumor-dataset-10-classes/data'
    SAVE_DIR = '/kaggle/working/output'

    IMAGE_SIZE = (299, 299)  # Inception requires 299x299
    BATCH_SIZE = 64
    EPOCHS = 20
    LEARNING_RATE = 0.0001
    NUM_CLASSES = 10

    if not os.path.exists(DATA_DIR):
        print(f"\n❌ Error: Data directory '{DATA_DIR}' not found!")
        return

    for subdir in ['train', 'val', 'test']:
        path = os.path.join(DATA_DIR, subdir)
        if os.path.exists(path):
            print(f"✅ Found: {path}")
        else:
            print(f"❌ Missing: {path}")
            return

    print("\n🏗️  Building Inception-v4 model...")
    model = InceptionV4Classifier(input_shape=(*IMAGE_SIZE, 3), num_classes=NUM_CLASSES)
    model.build_model(trainable=True)
    model.compile_model(learning_rate=LEARNING_RATE)

    print("\n📋 Model Architecture:")
    print("=" * 80)
    model.get_model_summary()
    model.count_parameters()

    trainer = BrainTumorTrainer(data_dir=DATA_DIR, model=model, batch_size=BATCH_SIZE, image_size=IMAGE_SIZE)
    trainer.setup_data_generators()
    trainer.train(epochs=EPOCHS, save_dir=SAVE_DIR)
    trainer.plot_training_history(save_dir=SAVE_DIR)
    trainer.evaluate_on_test(save_dir=SAVE_DIR)

    print("\n" + "=" * 80)
    print("✅ TRAINING COMPLETE!")
    print(f"📂 Results saved to: {SAVE_DIR}/")
    print("=" * 80)


if __name__ == "__main__":
    main()


2026-04-30 16:14:59.282242: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777565699.484491      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777565699.545668      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777565700.017348      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777565700.017400      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777565700.017404      57 computation_placer.cc:177] computation placer alr


🧠 Inception-v4 - BRAIN TUMOR CLASSIFICATION
Based on: Ishfaq et al. (2025) - Scientific Reports
Using PRE-AUGMENTED Data
✅ Found: /kaggle/input/datasets/fazilsiddiqui/brain-tumor-dataset-10-classes/data/train
✅ Found: /kaggle/input/datasets/fazilsiddiqui/brain-tumor-dataset-10-classes/data/val
✅ Found: /kaggle/input/datasets/fazilsiddiqui/brain-tumor-dataset-10-classes/data/test

🏗️  Building Inception-v4 model...


I0000 00:00:1777565728.057200      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777565728.063861      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


219055592/219055592 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
✅ Inception-v4 model compiled successfully
   Optimizer: Adam (lr=0.0001)

📋 Model Architecture:


Model: "InceptionV4_Classifier"

...

215/215 ━━━━━━━━━━━━━━━━━━━━ 882s 3s/step - accuracy: 0.6404 - loss: 1.0592 - precision: 0.8601 - recall: 0.5062 - val_accuracy: 0.9538 - val_loss: 0.1649 - val_precision: 0.9579 - val_recall: 0.9501 - learning_rate: 1.0000e-04
Epoch 2/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9801 - loss: 0.0629 - precision: 0.9829 - recall: 0.9780
Epoch 2: val_accuracy improved from 0.95380 to 0.97486, saving model to /kaggle/working/output/inception_v4_best.h5


215/215 ━━━━━━━━━━━━━━━━━━━━ 529s 2s/step - accuracy: 0.9801 - loss: 0.0629 - precision: 0.9829 - recall: 0.9780 - val_accuracy: 0.9749 - val_loss: 0.0818 - val_precision: 0.9791 - val_recall: 0.9718 - learning_rate: 1.0000e-04
Epoch 3/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9909 - loss: 0.0306 - precision: 0.9921 - recall: 0.9897
Epoch 3: val_accuracy improved from 0.97486 to 0.99355, saving model to /kaggle/working/output/inception_v4_best.h5


215/215 ━━━━━━━━━━━━━━━━━━━━ 530s 2s/step - accuracy: 0.9909 - loss: 0.0306 - precision: 0.9921 - recall: 0.9898 - val_accuracy: 0.9935 - val_loss: 0.0269 - val_precision: 0.9935 - val_recall: 0.9929 - learning_rate: 1.0000e-04
Epoch 4/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9942 - loss: 0.0189 - precision: 0.9948 - recall: 0.9937
Epoch 4: val_accuracy did not improve from 0.99355
215/215 ━━━━━━━━━━━━━━━━━━━━ 529s 2s/step - accuracy: 0.9942 - loss: 0.0189 - precision: 0.9948 - recall: 0.9937 - val_accuracy: 0.9851 - val_loss: 0.0638 - val_precision: 0.9861 - val_recall: 0.9847 - learning_rate: 1.0000e-04
Epoch 5/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9957 - loss: 0.0150 - precision: 0.9961 - recall: 0.9955
Epoch 5: val_accuracy did not improve from 0.99355
215/215 ━━━━━━━━━━━━━━━━━━━━ 527s 2s/step - accuracy: 0.9957 - loss: 0.0150 - precision: 0.9961 - recall: 0.9955 - val_accuracy: 0.9918 - val_loss: 0.0348 - val_precision: 0.9918 - val_recall: 0.991

215/215 ━━━━━━━━━━━━━━━━━━━━ 530s 2s/step - accuracy: 0.9982 - loss: 0.0076 - precision: 0.9983 - recall: 0.9982 - val_accuracy: 0.9973 - val_loss: 0.0140 - val_precision: 0.9973 - val_recall: 0.9973 - learning_rate: 5.0000e-05
Epoch 8/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9989 - loss: 0.0034 - precision: 0.9989 - recall: 0.9988
Epoch 8: val_accuracy did not improve from 0.99728
215/215 ━━━━━━━━━━━━━━━━━━━━ 528s 2s/step - accuracy: 0.9989 - loss: 0.0034 - precision: 0.9990 - recall: 0.9988 - val_accuracy: 0.9969 - val_loss: 0.0139 - val_precision: 0.9973 - val_recall: 0.9969 - learning_rate: 5.0000e-05
Epoch 9/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 1.0000 - loss: 6.6009e-04 - precision: 1.0000 - recall: 1.0000
Epoch 9: val_accuracy improved from 0.99728 to 0.99762, saving model to /kaggle/working/output/inception_v4_best.h5


215/215 ━━━━━━━━━━━━━━━━━━━━ 530s 2s/step - accuracy: 1.0000 - loss: 6.6024e-04 - precision: 1.0000 - recall: 1.0000 - val_accuracy: 0.9976 - val_loss: 0.0129 - val_precision: 0.9976 - val_recall: 0.9976 - learning_rate: 5.0000e-05
Epoch 10/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 1.0000 - loss: 4.7442e-04 - precision: 1.0000 - recall: 1.0000
Epoch 10: val_accuracy did not improve from 0.99762
215/215 ━━━━━━━━━━━━━━━━━━━━ 527s 2s/step - accuracy: 1.0000 - loss: 4.7552e-04 - precision: 1.0000 - recall: 1.0000 - val_accuracy: 0.9973 - val_loss: 0.0177 - val_precision: 0.9973 - val_recall: 0.9973 - learning_rate: 5.0000e-05
Epoch 11/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9996 - loss: 0.0020 - precision: 0.9996 - recall: 0.9996
Epoch 11: val_accuracy did not improve from 0.99762
215/215 ━━━━━━━━━━━━━━━━━━━━ 528s 2s/step - accuracy: 0.9996 - loss: 0.0020 - precision: 0.9996 - recall: 0.9996 - val_accuracy: 0.9901 - val_loss: 0.0361 - val_precision: 0.9908 - v